# CAMELS-ES: Time Series
***

***Author:** Chus Casado Rodríguez*<br>
***Date:** 22-06-2026*<br>

**Introducción:**<br>



**Outputs:**<br>


**To do**:<br>
* [] Filter stations with wrong catchment polygon and fix it.
* [] Probably the timestamps in CERRA are shifted one day (like EMO1).
* [x] How to trim the meteo time series at the start? Should I include one extra year as initial condition for the first discharge observation?

In [2]:
from tqdm.auto import tqdm
import json
import logging
logger = logging.getLogger(__name__)

import pandas as pd
import geopandas as gpd
from sklearn.model_selection import train_test_split

from ocab.config import Config
import ocab.variables as vars
from ocab.timeseries.utils import time_encoding
from ocab.utils.sampling import create_sample_file, create_period_file

## Configuration

In [12]:
cfg = Config('config_CAMELS_v200.yml')

# point layer
filename = 'stations.geojson'

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'

# questionnaire
url_form = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vS6FRrdMVZHu_JpG9e5wAWi-PNYWs3Hvqr-qmW7C8gbMHbBRHR1D8fomIjBAgNuhqFlNOhmnK6zJp-z/pub?output=csv'

# import timeseries metadata
with open('metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

## Data

### Stations

In [70]:
# load points
points = gpd.read_file(cfg.path_gis / filename).set_index('id')
print(f'no. points: {len(points):4}')

# identify basins
basins = points['basin'].unique()
print(f'no. basins: {len(basins):4}')

no. points: 1117
no. basins:   12


### Selection

I load and handle the answers to the questionnaire in the [website](https://casadoj.github.io/of_camels_and_beavers/).

In [50]:
# load answers to the online questionnaire
answers = pd.read_csv(url_form, parse_dates=True)
print(len(answers))

# rename columns
rename_cols = {
    'Marca temporal': 'timestamp', 
    'Station ID': 'ID', 
    'Hydrological regime': 'regime',
    'Start date (1st period)': 'start_1', 
    'End date (1st period) ': 'end_1',
    'Raise any other issue in the station attributes or time series. ': 'comments',
    # 'email', 
    'Is the catchment polygon correct?': 'catchment',
    'Is there a second period of high-quality data?': 'second_period',
    'Start date (2nd period)': 'start_2', 
    'End date (2nd period)': 'end_2',
}
answers.rename(columns=rename_cols, inplace=True)

779


***

In [5]:
# for basin in ['CANTABRICO', 'GALICIA COSTA', 'MIÑO-SIL', 'DUERO']:
#     missing = points[points['basin'] == basin].index.difference(answers['ID'])
#     if len(missing > 0):
#         print(basin, *missing)

***

In [51]:

# keep only selected stations
answers = answers[answers['ID'].isin(points.index)]
print('Raw data')
print(f'No. answers:\t\t{len(answers)}')
print(f'No. unique stations:\t{len(answers["ID"].unique())}')

# select stations with natural or semi-natural regimes
# if multiple answers, I take the majority vote
IDs = []
for ID in answers['ID'].unique():
    subset = answers[answers['ID'] == ID]
    if len(subset) > 1:
        if subset['regime'].value_counts().index[0] in ['Natural', 'Semi-natural']:
            IDs.append(ID)
    else:
        if subset['regime'].item() in ['Natural', 'Semi-natural']:
            IDs.append(ID)
mask_id = answers['ID'].isin(IDs)
mask_regime = answers['regime'].isin(['Natural', 'Semi-natural'])
answers = answers[mask_id & mask_regime]

print('\n(Semi)natural regime')
print(f'No. answers:\t\t{len(answers)}')
print(f'No. unique stations:\t{len(answers["ID"].unique())}')

# if duplicate stations, keep only the last answer
answers = answers.sort_values('timestamp').drop_duplicates('ID', keep='last')
answers.set_index('ID', inplace=True, drop=True)


# combine selected periods
def combine_periods(row, days: int = 365):
    if row['second_period'] == 'Yes':
        starts = [row['start_1'], row['start_2']]
        ends = [row['end_1'], row['end_2']]
    else:
        starts = [row['start_1']]
        ends = [row['end_1']]
    starts = pd.to_datetime(starts, dayfirst=True) - pd.Timedelta(days=days)
    ends = pd.to_datetime(ends, dayfirst=True)
    return pd.Series([starts, ends], index=['start_dates', 'end_dates'])
answers[['start_dates', 'end_dates']] = answers.apply(combine_periods, axis=1)

# drop some columns
answers.drop(columns=['timestamp', 'start_1', 'end_1', 'start_2', 'end_2'], inplace=True)

Raw data
No. answers:		777
No. unique stations:	767

(Semi)natural regime
No. answers:		501
No. unique stations:	495


## Export

### Samples and Periods

In [75]:
# define output folder
path_samples = cfg.path_dataset / 'selection'
path_samples.mkdir(exist_ok=True)
print(f'Samples will be saved in {path_samples}')

# keep points selected in the questionnaire
points = points.loc[points.index.intersection(answers.index)]

# divide basins in train, validation and test sets
train_set, temp = train_test_split(
    points,
    train_size=cfg.train_size,
    random_state=cfg.seed,
    stratify=points['basin']
)
val_set, test_set = train_test_split(
    temp,
    train_size=cfg.val_size / (1 - cfg.train_size),
    random_state=cfg.seed,
    stratify=temp['basin']
)
print(f'train size:\t\t{len(train_set)}')
print(f'validation size:\t{len(val_set)}')
print(f'test size:\t\t{len(test_set)}')

# organize basin IDs according to the sample
samples = {
    'train': train_set.sort_index().index.to_list(),
    'validation': val_set.sort_index().index.to_list(),
    'test': test_set.sort_index().index.to_list(),
}

# create sample files
for name, sample in samples.items():

    # export TXT file

    # all dataset
    create_sample_file(cfg, sample, path_samples / f'basins_{name}.txt')

    # per basin
    for basin in basins:
        sample_basin = points[points['basin'] == basin].index.intersection(sample).tolist()
        if len(sample_basin) > 0:
            create_sample_file(cfg, sample_basin, path_samples / f'{basin.lower()}_{name}.txt')

    # export PKL file
    create_period_file(cfg, sample, answers, path_samples / f'periods_{name}.pkl')

Samples will be saved in /home/casadoj/Data/CAMELS-ES/v2_0_0/selection
train size:		297
validation size:	99
test size:		99



### Time series

In [76]:
# define output folder
path_csv = cfg.path_timeseries / 'csv' #/ cfg.prefix
path_nc = cfg.path_timeseries / 'netcdf' #/ cfg.prefix
for path in [path_csv, path_nc]:
    path.mkdir(parents=True, exist_ok=True)
print(f'Time series will be saved in {cfg.path_timeseries}')

# process timeseries for each station
for ID in tqdm(answers.index, desc='points'):
    
    # DISCHARGE TIME SERIES
    # .....................
    try:
        dis = pd.read_parquet(path_in / 'discharge' / f'{ID}.parquet')
        dis.columns = ['discharge_cms']
        # compute specific discharge (mm/day)
        dis['discharge_mm'] = dis['discharge_cms'] / points.loc[ID, 'catch_skm'] * 86400 / 1000
        # round values
        dis = dis[dis.columns.intersection(vars.DECIMALS)].round(vars.DECIMALS)
    except Exception as e:
        logger.error(f'Loading discharge timeseries for station {ID:04d}: {e}')
        continue
    
    # METEOROLOGICAL TIME SERIES
    # ..........................
    meteo = pd.DataFrame()
    for dataset in ['ROCIO-IBEB', 'EMO1']:#, 'CERRA']:
        try:
            # read timeseries
            meteo_ts = pd.read_parquet(path_in / 'meteo' / dataset / f'{ID}.parquet').loc[ID]
            # rename variables
            meteo_ts.rename(columns=vars.RENAME, inplace=True, errors='ignore')
            # round_values
            meteo_ts = meteo_ts[meteo_ts.columns.intersection(vars.DECIMALS)].round(vars.DECIMALS)
            # add dataset suffix
            suffix = dataset.split('-')[0].lower()
            meteo_ts.columns = [f'{col}_{suffix}' for col in meteo_ts.columns]
            # correct dates
            if dataset == 'EMO1':
                meteo_ts.index = meteo_ts.index.date - pd.Timedelta(days=1)
            meteo_ts.index.name = 'date'
            meteo_ts.index = pd.to_datetime(meteo_ts.index)
            # concatenate
            meteo = pd.concat([meteo, meteo_ts], axis=1, sort=True)
        except Exception as e:
            logger.error(f'Loading {dataset} meteorological timeseries for station {ID}: {e}')
            continue
    
    # merge timeseries
    start = min(answers.loc[ID, 'start_dates'])
    end = max(answers.loc[ID, 'end_dates'])
    ts = pd.concat([dis.loc[start:end], meteo.loc[start:end]], axis=1, sort=True)

    # TEMPORAL ENCODERS
    # .................
    ts['year'] = ts.index.year
    ts['month'] = ts.index.month
    ts['month_sin'], ts['month_cos'] = time_encoding(ts['month'], period=12)
    ts['weekofyear'] = ts.index.isocalendar().week.astype('uint32')
    ts['woy_sin'], ts['woy_cos'] = time_encoding(ts['weekofyear'], period=52)
    ts['dayofyear'] = ts.index.dayofyear
    ts['doy_sin'], ts['doy_cos'] = time_encoding(ts['dayofyear'], period=365)
    ts['dayofweek'] = ts.index.isocalendar().day.astype('uint32')
    ts['dow_sin'], ts['dow_cos'] = time_encoding(ts['dayofweek'], period=7)

    # EXPORT
    # ......
    
    # export CSV file
    ts.to_csv(path_csv / f'{cfg.prefix}_{ID}.csv', index=True)

    # export NetCDF file
    ds = ts.to_xarray()
    ds.attrs['Timezone'] = metadata['Timezone']
    ds.attrs['Sources'] = metadata['Sources']
    for var in ds.data_vars:
        var_short = '_'.join(var.split('_')[:2])  # remove dataset suffix
        if var_short in metadata['variables']:
            ds[var].attrs['long_name'] = metadata['variables'][var_short]['long_name']
            ds[var].attrs['units'] = metadata['variables'][var_short]['units']
    ts.to_xarray().to_netcdf(path_nc / f'{cfg.prefix}_{ID}.nc')

Time series will be saved in /home/casadoj/Data/CAMELS-ES/v2_0_0/timeseries


points:   0%|          | 0/495 [00:00<?, ?it/s]